In [6]:
import pandas as pd
from scipy import stats

In [7]:
df_grid = pd.read_csv('Test History/cleaned_Ensemble_Grid_Optimization_Results.csv')
df_abc = pd.read_csv('Test History/cleaned_ABC_CV_RELM_Algo_2_2_20SN_100MI_Results.csv')

In [8]:
df_grid.describe()

,Hidden_Nodes,Lambda_Value,Solution_Size,Trial_Limit,Max_Iteration,Fold_ID,Seed,Accuracy,Precision,Recall,NPV,Specificity,F1-Score,F2-Score,Bal Accuracy,MCC
count,150.0,150.0000,0.0,0.0,0.0,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,30.0,0.0625,NaN,NaN,NaN,2.000000,115.500000,0.771847,0.812613,0.701868,0.744408,0.840038,0.751544,0.720615,0.770953,0.549364
std,0.0,0.0000,NaN,NaN,NaN,1.418951,8.684438,0.049339,0.061437,0.068415,0.049006,0.058440,0.056168,0.062718,0.049434,0.099385
min,30.0,0.0625,NaN,NaN,NaN,0.000000,101.000000,0.650794,0.678571,0.548387,0.631579,0.687500,0.607143,0.570470,0.649194,0.304912
25%,30.0,0.0625,NaN,NaN,NaN,1.000000,108.000000,0.734375,0.769231,0.656250,0.710526,0.812500,0.714286,0.678519,0.734375,0.470987
50%,30.0,0.0625,NaN,NaN,NaN,2.000000,115.500000,0.777778,0.814815,0.709677,0.742857,0.843750,0.752049,0.723684,0.776179,0.555444
75%,30.0,0.0625,NaN,NaN,NaN,3.000000,123.000000,0.809524,0.857143,0.750000,0.777778,0.875000,0.798276,0.768590,0.808972,0.623250
max,30.0,0.0625,NaN,NaN,NaN,4.000000,130.000000,0.859375,0.956522,0.843750,0.843750,0.968750,0.852459,0.843750,0.859375,0.726117


In [9]:
df_abc.describe()

,Hidden_Nodes,Lambda_Value,Solution_Size,Trial_Limit,Max_Iteration,Fold_ID,Seed,Accuracy,Precision,Recall,NPV,Specificity,F1-Score,F2-Score,Bal Accuracy,MCC
count,150.0,150.0000,150.0,150.0,150.0,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,30.0,0.0625,20.0,10.0,100.0,2.000000,115.500000,0.790971,0.843011,0.710208,0.756474,0.869577,0.769388,0.732467,0.789893,0.589496
std,0.0,0.0000,0.0,0.0,0.0,1.418951,8.684438,0.044815,0.054710,0.067039,0.045676,0.049066,0.053292,0.061131,0.044960,0.089555
min,30.0,0.0625,20.0,10.0,100.0,0.000000,101.000000,0.682540,0.724138,0.548387,0.650000,0.750000,0.629630,0.578231,0.680444,0.374743
25%,30.0,0.0625,20.0,10.0,100.0,1.000000,108.000000,0.765625,0.806762,0.656250,0.725000,0.843750,0.727273,0.686112,0.761241,0.533600
50%,30.0,0.0625,20.0,10.0,100.0,2.000000,115.500000,0.796875,0.846154,0.718750,0.763158,0.875000,0.779661,0.740095,0.796875,0.596377
75%,30.0,0.0625,20.0,10.0,100.0,3.000000,123.000000,0.822173,0.887821,0.750000,0.787879,0.906250,0.804839,0.775847,0.820817,0.657919
max,30.0,0.0625,20.0,10.0,100.0,4.000000,130.000000,0.875000,0.960000,0.903226,0.900000,0.969697,0.875000,0.891720,0.873488,0.761982


In [10]:
metrics = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'Bal Accuracy', 'MCC']

def calculate_lcb(df, metrics):

    grouped = df.groupby('Seed')[metrics]
    return grouped.mean() - grouped.std()

lcb_grid = calculate_lcb(df_grid, metrics)
lcb_abc = calculate_lcb(df_abc, metrics)

# 3. Alignment
aligned_seeds = lcb_grid.index.intersection(lcb_abc.index)
lcb_grid, lcb_abc = lcb_grid.loc[aligned_seeds], lcb_abc.loc[aligned_seeds]

results = []
for metric in metrics:
    # Extract aligned arrays for the current metric
    stat_grid = lcb_grid[metric].values
    stat_abc = lcb_abc[metric].values

    # --- Central Tendency Testing ---
    w_stat, p_val_w = stats.wilcoxon(stat_abc, stat_grid)
    t_stat, p_val_t = stats.ttest_rel(stat_abc, stat_grid)

    mean_grid = stat_grid.mean()
    mean_abc = stat_abc.mean()

    # --- Variance Testing (Cross-Seed Stability) ---
    # Calculate sample standard deviation across seeds
    std_grid = stat_grid.std(ddof=1)
    std_abc = stat_abc.std(ddof=1)

    # Levene's test for equality of variances
    # Null Hypothesis: The variance of LCB across seeds is equal for both methods.
    stat_levene, p_val_levene = stats.levene(stat_abc, stat_grid, center='median')

    # --- Result Aggregation ---
    results.append({
        'Metric': metric,
        'Grid_Mean_LCB': mean_grid,
        'ABC_Mean_LCB': mean_abc,
        'Wilcoxon_p_value': p_val_w,
        'Significant_ABC_Better': (p_val_w < 0.05) and (mean_abc > mean_grid),

        # New Stability Metrics
        'Grid_Std_LCB': std_grid,
        'ABC_Std_LCB': std_abc,
        'Levene_p_value': p_val_levene,
        'Significant_Variance_Diff': (p_val_levene < 0.05),
        'More_Stable_Method': 'ABC' if (std_abc < std_grid and p_val_levene < 0.05)
                              else ('Grid' if (std_grid < std_abc and p_val_levene < 0.05) else 'Inconclusive/Equal')
    })

df_results = pd.DataFrame(results)
df_results

,Metric,Grid_Mean_LCB,ABC_Mean_LCB,Wilcoxon_p_value,Significant_ABC_Better,Grid_Std_LCB,ABC_Std_LCB,Levene_p_value,Significant_Variance_Diff,More_Stable_Method
0,Accuracy,0.721144,0.744555,0.000283,True,0.023191,0.017231,0.089920,False,Inconclusive/Equal
1,Precision,0.752751,0.788065,0.000071,True,0.030005,0.025117,0.258224,False,Inconclusive/Equal
2,Recall,0.632948,0.643084,0.244946,False,0.032195,0.035030,0.972824,False,Inconclusive/Equal
3,Specificity,0.784408,0.821661,0.000123,True,0.030666,0.027689,0.543563,False,Inconclusive/Equal
4,F1-Score,0.693390,0.714554,0.002367,True,0.024928,0.022378,0.418941,False,Inconclusive/Equal
5,Bal Accuracy,0.720099,0.743344,0.000380,True,0.023052,0.017308,0.100988,False,Inconclusive/Equal
6,MCC,0.447677,0.496677,0.000209,True,0.047880,0.034440,0.071166,False,Inconclusive/Equal
